# Mini Project: Risk–Return Portfolio Optimization with GA & PSO

**Student template** — complete all `TODO` sections. The dataset contains **simulated** prices, not real historical stock prices. The primary task is implementing and understanding GA and PSO.

## 0. Setup
Upload `simulated_stock_prices.csv` to the same directory as this notebook. In Colab, upload the CSV into the working directory.

In [ ]:
%pip -q install pygad pyswarms

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import pygad
import pyswarms as ps

SEED = 2026
np.random.seed(SEED)

## 1. Read data and calculate daily returns (Task 1)

In [ ]:
prices = pd.read_csv("simulated_stock_prices.csv", parse_dates=["Date"]).set_index("Date")
print(prices.shape)
display(prices.head())
print("Missing values:", prices.isna().sum().to_dict())
returns = prices.pct_change().dropna()
mu = returns.mean().to_numpy() * 252
Sigma = returns.cov().to_numpy() * 252
assets = list(prices.columns)
print("Annualized expected returns:", dict(zip(assets, np.round(mu,4))))
print("Annualized covariance matrix:\n", Sigma)

## 2. Objective and feasibility
We maximize `expected_return − lambda × variance`. The optimizer can search nonnegative raw coordinates. Normalize the raw vector to make its entries sum to one. Zero vectors are mapped to equal weights. **Do not change this rule between GA and PSO.**

In [ ]:
LAMBDA = 3.0
n_assets = len(assets)

def normalize_weights(raw):
    raw = np.asarray(raw, dtype=float)
    raw = np.clip(raw, 0.0, None)
    total = raw.sum(axis=-1, keepdims=True)
    return np.divide(raw, total, out=np.full_like(raw, 1.0/raw.shape[-1]), where=total>1e-12)

def portfolio_metrics(raw, risk_aversion=LAMBDA):
    w = normalize_weights(raw)
    exp_return = float(w @ mu)
    variance = float(w @ Sigma @ w)
    volatility = float(np.sqrt(max(variance,0)))
    score = exp_return - risk_aversion * variance
    return {"return":exp_return,"variance":variance,"volatility":volatility,"score":score}

assert np.isclose(normalize_weights(np.ones(n_assets)).sum(),1)
print(portfolio_metrics(np.ones(n_assets)))

## 3. Genetic Algorithm (Task 2)
**TODO:** Complete the fitness function and the `pygad.GA(...)` configuration. Use floating-point genes, 30 solutions and 100 generations. Suggested settings: parent selection `sss`, crossover `single_point`, mutation `random`, 10% genes, gene space [0,1]. The GA maximizes fitness.

In [ ]:
def ga_fitness(ga_instance, solution, solution_idx):
    # TODO: return the risk-return objective score for this chromosome
    raise NotImplementedError("Complete GA fitness")

# TODO: instantiate and run pygad.GA with:
# num_generations=100, sol_per_pop=30, num_genes=n_assets,
# num_parents_mating=10, gene_space={"low":0.0,"high":1.0},
# fitness_func=ga_fitness, random_seed=SEED,
# parent_selection_type="sss", crossover_type="single_point",
# mutation_type="random", mutation_percent_genes=10
# Record runtime and best solution.
# ga = ...
# ga.run()
# ga_solution, ga_fitness_best, _ = ga.best_solution()
# ga_weights = normalize_weights(ga_solution)

## 4. Particle Swarm Optimization (Task 3)
**TODO:** Complete the PSO cost function. `pyswarms` minimizes costs, so use the **negative** of the same objective score. Your function receives an `(n_particles, 5)` array and must return an `(n_particles,)` array.

In [ ]:
def pso_cost(X):
    # TODO: compute feasible weights and vectorized portfolio scores for every particle
    # Hint: expected_return = W @ mu
    # Hint: variance = np.einsum("ij,jk,ik->i", W, Sigma, W)
    # Return negative scores because PSO minimizes.
    raise NotImplementedError("Complete PSO cost")

# TODO: initialize ps.single.GlobalBestPSO:
# n_particles=30, dimensions=n_assets, options={"c1":1.5,"c2":1.5,"w":0.7},
# bounds=(np.zeros(n_assets), np.ones(n_assets))
# optimizer.optimize(pso_cost, iters=100, verbose=False)
# Record runtime and best position.
# pso_weights = normalize_weights(best_position)

## 5. Results and visualizations (Task 4)
Complete the algorithm sections above first. Then uncomment and run this cell.

In [ ]:
# TODO: uncomment after defining ga_weights, pso_weights and runtimes
# portfolios = {"Equal Weight": np.ones(n_assets)/n_assets, "GA":ga_weights, "PSO":pso_weights}
# rows = []
# for name, weights in portfolios.items():
#     rows.append({"Method":name, **portfolio_metrics(weights)})
# results = pd.DataFrame(rows).set_index("Method")
# display(results)
# pd.DataFrame(portfolios,index=assets).plot(kind="bar",figsize=(9,4))
# plt.ylabel("Portfolio weight");plt.tight_layout();plt.show()
# plt.plot(ga.best_solutions_fitness,label="GA: best fitness (maximize)")
# plt.plot(-np.asarray(optimizer.cost_history),label="PSO: negative best cost (maximize)")
# plt.xlabel("Iteration / generation");plt.ylabel("Objective score");plt.legend();plt.show()

## 6. Robustness and risk-aversion sensitivity (Tasks 4–5)
**TODO:** Repeat GA and PSO for seeds `[11,22,33]` and summarize objective scores. Repeat for `lambda = 1, 3, 8`. Ensure both fitness functions accept the chosen lambda. Explain whether increasing lambda lowers portfolio variance in your results.

In [ ]:
seeds = [11,22,33]
lambda_values = [1.0,3.0,8.0]
# TODO: implement repeated runs and a results table
# Suggested columns: method, seed, lambda, objective, return, volatility, runtime

## 7. Discussion (Task 5)
Answer in your own words:

1. What do the GA chromosome and PSO particle represent?
2. How does normalization enforce the portfolio constraints?
3. What are the roles of selection, crossover and mutation in GA?
4. What are pbest, gbest and velocity in PSO?
5. Which algorithm performed better on your runs, and is that conclusion stable?
6. What changed when λ increased from 1 to 8?
7. Why does the optimized portfolio not necessarily have the highest possible return?